# Cycle 5 : Test oversampling


In [1]:
### Import des modules
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import utils
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [2]:
data = pd.read_csv('data/rafined/employees.csv', sep=',')

In [4]:
train_data = utils.split_train_data(data, 'a_quitte_l_entreprise')

In [5]:
from sklearn.preprocessing import RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('robust_scaler', RobustScaler(), ['revenu_mensuel']),
        ('minmax_scaler', MinMaxScaler(), ['age']),
        ('encoder', OneHotEncoder(), ['statut_marital', 'departement', 'poste', 'domaine_etude']),
    ],
    remainder='passthrough'
)

In [8]:
from sklearn.model_selection import FixedThresholdClassifier
from technova_features import TechNovaFeatureEngineering
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

random_forest_model = RandomForestClassifier(random_state=42)

threshold_model = FixedThresholdClassifier(
    estimator=random_forest_model,
    threshold=0.3,
    response_method="predict_proba"
)

pipeline = Pipeline([
    ('features', TechNovaFeatureEngineering()),
    ('preprocessor', preprocessor),
    ('sampling', SMOTE(random_state=42)),
    ('model', threshold_model),
])

utils.benchmark(pipeline, train_data)

CrossValidation Results : 
{'fit_time': array([0.1786201 , 0.16992307, 0.17348003, 0.17501092, 0.1736989 ]), 'score_time': array([0.01248288, 0.01189518, 0.0115881 , 0.01254106, 0.01188493]), 'test_recall': array([0.675     , 0.66666667, 0.61538462, 0.625     , 0.75      ]), 'test_f1': array([0.57446809, 0.57142857, 0.48484848, 0.4950495 , 0.56074766])}
Recall moyen : 0.6664102564102564
F1-Score moyen : 0.5373084619770673
Training Résults : 
Recall moyen : 0.48717948717948717
F1-Score moyen : 0.40425531914893614
[[219  36]
 [ 20  19]]


In [ ]:
utils.train(pipeline, preprocessor, train_data, 0.5)